## Import package

In [1]:
import sys
sys.path.append('./script')
from pyomo_solving import *
from ET_optme import *
import pandas as pd
import cobra
import gurobipy
import json
import multiprocessing
import os
from multiprocessing import Pool
from sympy import subsets
import pandas as pd
import matplotlib.pyplot as plt
import re

## setting path for models and data, example: C.glutamicum --->L_Lyscine

In [2]:
model_file = "./file/iCZ870.json"
model0_file = "./file/iCZ870.json"
reaction_kcat_MW_file = './file/reaction_kcat_MW_modify_n.csv'
dictionarymodel_path = './results/etcz870.json'
reaction_g0_file=os.path.join('./file/iCZ870_g0_n.csv')
metabolites_lnC_file = os.path.join('file/iCZ870_met.txt')
path_results = './results/icz870_lysine_ETFSEOF'
savepath = './results/picture'
inputdic = {"model":'iCZ870.json',
"substrate":"EX_cpd00027_e_reverse",
"biomass": "EX_biomass_c",
"product": "EX_cpd00054_e",
"taskname": "ET-FSEOF",
"mode":"SET",
"oxygenstate":"aerobic"}
# save task.json
with open('./results/task_fseof.json', 'w') as json_file:
    json.dump(inputdic, json_file, indent=4)
path_strain = 'iCW'

## invoke ET-FSEOF and run the task 

## 1.enzyme-thermo Model Construction ( this step may takes 10 minutes)

In [3]:
# Read the model
model = cobra.io.load_json_model(model_file)

# Load the model data into Get_Concretemodel_Need_Data
Concretemodel_Need_Data = Get_Concretemodel_Need_Data(model_file)

model0 = cobra.io.load_json_model(model0_file)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2788620
Academic license 2788620 - for non-commercial use only - registered to 90___@qq.com


In [4]:
# Store the protein-centered model as a dictionary
dictionary_model = json_load(dictionarymodel_path)
dictionary_model.keys()
rname3 = []

# Store the protein-centered dictionary into Get_Concretemodel_Need_Data
get_dictionarymodel_data2(dictionary_model, Concretemodel_Need_Data, rname3)

# Load thermodynamic data into Get_Concretemodel_Need_Data
Get_Concretemodel_Need_Data_g0(Concretemodel_Need_Data, reaction_g0_file, metabolites_lnC_file, reaction_kcat_MW_file)

Concretemodel_Need_Data['reaction_g0']['g0'] = Concretemodel_Need_Data['reaction_g0']['g0'].replace(0, np.nan)
Concretemodel_Need_Data['reaction_g0'].dropna(subset=['g0'], inplace=True)
#
# Concretemodel_Need_Data['reaction_g0'].at['AIRC3_reverse', 'g0'] = 0
# Concretemodel_Need_Data['reaction_g0'].at['ATPS4rpp_reverse_num2', 'g0'] = 0

Inc = Concretemodel_Need_Data['metabolites_lnC']
for i in Concretemodel_Need_Data['metabolite_list']:
    if i not in Inc.index:
        Inc.loc[i, 'lnClb'] = -14.508658
        Inc.loc[i, 'lnCub'] = -3.912023

Concretemodel_Need_Data['metabolites_lnC'] = Inc

# Modify the oxygen conditions of the model
if inputdic['oxygenstate'] == 'aerobic':
    model.reactions.get_by_id('EX_cpd00007_e_reverse').upper_bound = 1000
if inputdic['oxygenstate'] == 'micro_aerobic':
    model.reactions.get_by_id('EX_cpd00007_e_reverse').upper_bound = 2
if inputdic['oxygenstate'] == 'anaerobic':
    model.reactions.get_by_id('EX_cpd00007_e_reverse').upper_bound = 0


## 2.Algorithm :  ET-FSEOF

In [5]:
# Calculate the maximum production rate
product,objvalue2 = calculate_product_fseof(Concretemodel_Need_Data,inputdic,model)

ERROR: Solver (gurobi) returned non-zero return code (1)
ERROR: Solver log: Set parameter WLSAccessID Set parameter WLSSecret Set
parameter LicenseID to value 2788620 Academic license 2788620 - for non-
commercial use only - registered to 90___@qq.com Error reading LP format file
/tmp/tmpijtvzcyx.pyomo.lp at line 87924 Malformed term in expression
Neighboring tokens: " nan x19849 <= 0 c_u_x19850_: +1 x1307 "

    Traceback (most recent call last):
      File "<stdin>", line 5, in <module> File "/root/miniconda3/envs/ET-
      OptME/lib/python3.10/site-
      packages/pyomo/solvers/plugins/solvers/GUROBI_RUN.py", line 66, in
      gurobi_run
        model = read(model_file)
                ^^^^^^^^^^^^^^^^
      File "src/gurobipy/gurobi.pxi", line 3779, in gurobipy.read File
      "src/gurobipy/gurobi.pxi", line 117, in gurobipy.gurobi.read
    gurobipy.GurobiError: Unable to read file


ApplicationError: Solver (gurobi) did not exit normally

In [6]:
# Calculating the growth rate under the maximum production rate.
FSEOFdf,reactiondf = biomass(product,inputdic,Concretemodel_Need_Data,objvalue2,model,model0)

0.48243259499164753
0.48243259499164753
0.45671013424072276
0.09753991449669797
0.4294110669893727
0.4366491232972335
0.4366491232972335
0.41497382070802785
0.09728040942819073
0.39012097696895087
0.39086565188405326
0.39086565188405326
0.37323757432103355
0.0976455658730361
0.35076855576839616
0.3446733993342415
0.3446733993342415
0.3311286490044045
0.10356111836115509
0.3208525916598071
0.2988899277804441
0.2988899277804441
0.2893923628867754
0.11003635011056874
0.2915192331668831
0.2506843285816095
0.2506843285816095
0.23832432411673318
0.11621639783110098
0.24088506027344767
0.1994055902602643
0.1994055902602643
0.1852307194540423
0.12237909148073807
0.17607342083603789
0.1481268519389193
0.1481268519389193
0.13213711125943284
0.13011099241099638
0.11539184327288114
0.09639026773970506
0.09639026773970506
0.07856945798693073
0.14361718559297654
0.06486802418164028
0.043597941738332785
0.043597941738332785
0.013233329107538105
0.15149696871351498
0.003434294495323896


## Analyze the result and generate table

In [7]:
columns = FSEOFdf.columns
sorted_columns = sorted(columns, key=lambda x: get_sort_key(x, FSEOFdf))
sorted_FSEOFdf = FSEOFdf[sorted_columns]
FSEOFdf = sorted_FSEOFdf[['gene'] + [col for col in sorted_columns if col != 'gene']]
#  determine the manipulation strategies
FSEOFdf =detail(FSEOFdf)
#  get the mean flux
FSEOFdf =result(FSEOFdf)
# save the result table 
output_fseof(path_results,inputdic,reactiondf,FSEOFdf)

In [8]:
# show the key results 
FSEOFdf[['gene', 'manipulations']]

,gene,manipulations
0,CEY17_RS00005 and CEY17_RS00015 and CEY17_RS00...,down
1,CEY17_RS02725 and CEY17_RS02765 and CEY17_RS02...,down
2,CEY17_RS02795 and CEY17_RS02800 and CEY17_RS03...,down
3,CEY17_RS13790,down
4,CEY17_RS05620 and CEY17_RS05625,unchanged
...,...,...
697,CEY17_RS14500,unchanged
698,CEY17_RS14030,unchanged
699,CEY17_RS10365,unchanged
700,CEY17_RS15910,unchanged
